# Bengali LLM Benchmarks for Hate Speech Classification## Comparing Zero-Shot Bengali LLMs vs Our Consistency-Constrained MTL Model**Benchmarked Models:**1. **TigerLLM-1B-it** — State-of-the-art Bengali instruct LLM (ACL 2025)2. **TituLLM-1B** — First dedicated Bengali pretrained LLM (Llama-3.2 based)3. **BongLLaMA-3B** — Largest Bengali instruct model (Llama-3.2 fine-tune)**Metrics:** Macro F1 (Type, Target, Severity), Consistency Violation Rate (CVR), Latency

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepieceimport osimport jsonimport timeimport globimport reimport gcimport torchimport pandas as pdimport numpy as npfrom tqdm.auto import tqdmfrom sklearn.metrics import f1_scorefrom transformers import AutoTokenizer, AutoModelForCausalLMimport warningswarnings.filterwarnings('ignore')DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'OUTPUT_DIR = '/kaggle/working'RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')os.makedirs(RESULTS_DIR, exist_ok=True)print(f'Device: {DEVICE}')if torch.cuda.is_available():    print(f'GPU: {torch.cuda.get_device_name(0)}')    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---## 1. Load Evaluation Data

In [ ]:
# Auto-detect test.json or fall back to train.jsontest_paths = glob.glob('/kaggle/input/**/test.json', recursive=True)train_paths = glob.glob('/kaggle/input/**/train.json', recursive=True)if test_paths:    df_test = pd.read_json(test_paths[0])    print(f'Loaded test set: {test_paths[0]}')elif train_paths:    print('test.json not found. Sampling from train.json instead.')    df_test = pd.read_json(train_paths[0])else:    from datasets import load_dataset    dataset = load_dataset('aridhasan/BanglaMultiHate')    df_test = dataset['test'].to_pandas()# ── CRITICAL: Remap old taxonomy labels ──_LABEL_REMAP = {'Profane': 'Abusive', 'Sexism': 'Gender Hate'}df_test['type_of_hate'] = df_test['type_of_hate'].fillna('None').astype(str).str.strip().replace(_LABEL_REMAP)df_test['target_of_hate'] = df_test['target_of_hate'].fillna('None').astype(str).str.strip()df_test['severity_of_hate'] = df_test['severity_of_hate'].astype(str).str.strip()# Sample for evaluation (500 samples for speed)df_eval = df_test.sample(n=min(500, len(df_test)), random_state=42).reset_index(drop=True)print(f'Evaluation samples: {len(df_eval)}')print(f'Type distribution:\n{df_eval["type_of_hate"].value_counts().to_string()}')

---## 2. Shared Metrics & Parsing

In [ ]:
def check_consistency_violation(type_pred, target_pred, sev_pred):    """Returns True if prediction violates consistency rules."""    if type_pred == 'None':        if target_pred != 'None' or sev_pred != 'Little to None':            return True    return Falsedef extract_labels_from_json(text):    """Extract structured labels from LLM JSON output."""    try:        match = re.search(r'\{.*?\}', text, re.DOTALL)        if match:            parsed = json.loads(match.group(0))            return (                parsed.get('type_of_hate', 'None'),                parsed.get('target_of_hate', 'None'),                parsed.get('severity_of_hate', 'Little to None')            )    except:        pass    return 'None', 'None', 'Little to None'VALID_TYPES = {'None', 'Abusive', 'Political Hate', 'Religious Hate', 'Gender Hate'}VALID_TARGETS = {'None', 'Individual', 'Organization', 'Community', 'Society'}VALID_SEVERITIES = {'Little to None', 'Mild', 'Severe'}def normalize_prediction(t, trg, s):    """Clamp predictions to valid label set."""    if t not in VALID_TYPES: t = 'None'    if trg not in VALID_TARGETS: trg = 'None'    if s not in VALID_SEVERITIES: s = 'Little to None'    return t, trg, sdef build_zero_shot_prompt(comment):    return f"""নিচের মন্তব্যটি বিশ্লেষণ করুন এবং এর 'Hate Type', 'Target', এবং 'Severity' নির্ধারণ করুন।অপশনসমূহ:Hate Type: None, Abusive, Political Hate, Religious Hate, Gender HateTarget: None, Individual, Organization, Community, SocietySeverity: Little to None, Mild, Severeমন্তব্য: \"{comment}\"শুধুমাত্র নিচের JSON ফরম্যাটে উত্তর দিন:{{  "type_of_hate": "...",  "target_of_hate": "...",  "severity_of_hate": "..."}}"""def run_benchmark(model, tokenizer, model_name, max_new_tokens=80):    """Run zero-shot benchmark on a loaded model."""    preds_type, preds_target, preds_sev = [], [], []    violations = 0    start_time = time.time()        for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc=f'{model_name}'):        prompt = build_zero_shot_prompt(row['comment'])        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)                with torch.no_grad():            outputs = model.generate(                **inputs,                max_new_tokens=max_new_tokens,                temperature=0.1,                do_sample=True,                pad_token_id=tokenizer.eos_token_id            )                response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)        t, trg, s = extract_labels_from_json(response)        t, trg, s = normalize_prediction(t, trg, s)                if check_consistency_violation(t, trg, s):            violations += 1                preds_type.append(t)        preds_target.append(trg)        preds_sev.append(s)        elapsed = time.time() - start_time    latency = elapsed / len(df_eval)        return preds_type, preds_target, preds_sev, violations, latencyprint('Shared utilities loaded.')

---## 3. Benchmark A: TigerLLM-1B-itState-of-the-art Bengali instruct LLM (1B parameters, ACL 2025)

In [ ]:
MODEL_ID = "md-nishat-008/TigerLLM-1B-it"print(f"Loading {MODEL_ID}...")tokenizer_tiger = AutoTokenizer.from_pretrained(MODEL_ID)model_tiger = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")model_tiger.eval()preds_type_tiger, preds_target_tiger, preds_sev_tiger, violations_tiger, latency_tiger = \    run_benchmark(model_tiger, tokenizer_tiger, "TigerLLM-1B")# Free VRAMdel model_tiger, tokenizer_tigergc.collect()torch.cuda.empty_cache()print(f"TigerLLM done. Violations: {violations_tiger}/{len(df_eval)}, Latency: {latency_tiger:.2f}s/sample")

---## 4. Benchmark B: TituLLM-1BFirst dedicated Bengali pretrained LLM (1B parameters, Llama-3.2 based)

In [ ]:
MODEL_ID = "Hishab/tituLLM-1B"print(f"Loading {MODEL_ID}...")tokenizer_titu = AutoTokenizer.from_pretrained(MODEL_ID)model_titu = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")model_titu.eval()# TituLLM is pretrained (not instruct-tuned), so it may not follow prompts wellpreds_type_titu, preds_target_titu, preds_sev_titu, violations_titu, latency_titu = \    run_benchmark(model_titu, tokenizer_titu, "TituLLM-1B")del model_titu, tokenizer_titugc.collect()torch.cuda.empty_cache()print(f"TituLLM done. Violations: {violations_titu}/{len(df_eval)}, Latency: {latency_titu:.2f}s/sample")

---## 5. Benchmark C: BongLLaMA-3B-InstructLargest Bengali instruct model (3B parameters, Llama-3.2 fine-tuned)

In [ ]:
MODEL_ID = "BanglaLLM/bangla-llama-3.2-3b-instruct"print(f"Loading {MODEL_ID}...")tokenizer_bong = AutoTokenizer.from_pretrained(MODEL_ID)model_bong = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")model_bong.eval()preds_type_bong, preds_target_bong, preds_sev_bong, violations_bong, latency_bong = \    run_benchmark(model_bong, tokenizer_bong, "BongLLaMA-3B")del model_bong, tokenizer_bonggc.collect()torch.cuda.empty_cache()print(f"BongLLaMA done. Violations: {violations_bong}/{len(df_eval)}, Latency: {latency_bong:.2f}s/sample")

---## 6. Results Comparison

In [ ]:
true_type = df_eval['type_of_hate'].tolist()true_target = df_eval['target_of_hate'].tolist()true_sev = df_eval['severity_of_hate'].tolist()def calculate_metrics(p_type, p_target, p_sev, viols, latency, name):    type_f1 = f1_score(true_type, p_type, average='macro', zero_division=0)    target_f1 = f1_score(true_target, p_target, average='macro', zero_division=0)    sev_f1 = f1_score(true_sev, p_sev, average='macro', zero_division=0)    avg_f1 = (type_f1 + target_f1 + sev_f1) / 3    cvr = viols / len(p_type) * 100    print(f"\n{'='*60}")    print(f"  {name}")    print(f"{'='*60}")    print(f"  Hate Type F1:       {type_f1:.4f}")    print(f"  Target F1:          {target_f1:.4f}")    print(f"  Severity F1:        {sev_f1:.4f}")    print(f"  Avg Macro F1:       {avg_f1:.4f}")    print(f"  CVR:                {cvr:.2f}% ({viols}/{len(p_type)})")    print(f"  Latency:            {latency:.2f}s/sample")    return {        'model': name,        'type_f1': round(type_f1, 4),        'target_f1': round(target_f1, 4),        'sev_f1': round(sev_f1, 4),        'avg_f1': round(avg_f1, 4),        'cvr_pct': round(cvr, 2),        'violations': viols,        'total_samples': len(p_type),        'latency_s': round(latency, 3)    }r1 = calculate_metrics(preds_type_tiger, preds_target_tiger, preds_sev_tiger,                        violations_tiger, latency_tiger, "TigerLLM-1B-it")r2 = calculate_metrics(preds_type_titu, preds_target_titu, preds_sev_titu,                        violations_titu, latency_titu, "TituLLM-1B")r3 = calculate_metrics(preds_type_bong, preds_target_bong, preds_sev_bong,                        violations_bong, latency_bong, "BongLLaMA-3B-Instruct")# Save resultsresults = [r1, r2, r3]with open(os.path.join(RESULTS_DIR, 'llm_benchmark_results.json'), 'w') as f:    json.dump(results, f, indent=2)print(f"\nResults saved to {RESULTS_DIR}/llm_benchmark_results.json")# Summary tableprint(f"\n\n{'='*80}")print(f"  SUMMARY TABLE")print(f"{'='*80}")print(f"  {'Model':<30} {'Avg F1':>8} {'CVR':>8} {'Latency':>10}")print(f"  {'-'*30} {'-'*8} {'-'*8} {'-'*10}")for r in results:    print(f"  {r['model']:<30} {r['avg_f1']:>8.4f} {r['cvr_pct']:>7.2f}% {r['latency_s']:>8.3f}s")print(f"  {'-'*30} {'-'*8} {'-'*8} {'-'*10}")print(f"  {'Ours (MTL+Consistency, 110M)':<30} {'TBD':>8} {'~0.00':>7}% {'~0.01':>8}s")print(f"{'='*80}")